# Retail Sales Analytics Lakehouse

This notebook loads UCI retail transactions into Delta tables, checks the totals, and explores sales and repeat purchasing. Run `02_smoke_checks` first and keep `00_transformations` in the same folder.

The earlier full-data run returned 539,392 eligible lines and GBP 9,769,872.054 net sales. This revision includes later setup and query cleanup and still needs a fresh Databricks run. The earlier results are recorded in `reports/notebook-export-evidence.json`.

Full refresh replaces six project tables, one at a time. Use a dedicated schema, run one refresh at a time, and rerun from the beginning after a failure.


In [ ]:
%run ./00_transformations


## Configuration

Set the catalog, schema, and exact uploaded CSV path below. Start with the sample. Review the six target names before setting `ALLOW_FULL_REFRESH = True`. Later SQL cells use the catalog/schema selected during ingestion.


In [ ]:
import re
from decimal import Decimal

CATALOG = "workspace"
SCHEMA = "retail_learning"
INPUT_PATH = "/Volumes/workspace/retail_learning/retail_input/online_retail_sample.csv"
ALLOW_FULL_REFRESH = False

TARGET_TABLES = (
    "retail_bronze", "retail_silver", "retail_rejects",
    "retail_gold_daily", "retail_gold_product", "retail_gold_customer",
)

def validate_settings(catalog, schema, input_path):
    for label, value in [("catalog", catalog), ("schema", schema)]:
        if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
            raise ValueError(f"{label} must be a nonblank simple identifier without spaces")
    if not schema.startswith("retail_"):
        raise ValueError("Use a dedicated schema beginning with retail_")
    if not isinstance(input_path, str) or not re.fullmatch(r"/Volumes/[^/]+/[^/]+/[^/]+/.+\.csv", input_path):
        raise ValueError("Supply an exact CSV file path under a Unity Catalog volume")
    if any(part in (".", "..") for part in input_path.split("/")) or any(x in input_path for x in ("*", "?", "[", "]")):
        raise ValueError("Use a single explicit file path, without wildcards or traversal")

validate_settings(CATALOG, SCHEMA, INPUT_PATH)
namespace = f"`{CATALOG}`.`{SCHEMA}`"

def table(name):
    if name not in TARGET_TABLES:
        raise ValueError("Unexpected project table name")
    return f"{namespace}.`{name}`"

print("Target schema:", namespace)
print("Input file:", INPUT_PATH)
print("Tables to replace:", TARGET_TABLES)
print("Full refresh enabled:", ALLOW_FULL_REFRESH)


## Bronze ingestion

Check the source columns and row count before replacing Bronze. The row-count check adds a CSV read, so keep the input file unchanged during the run.


In [ ]:
if ALLOW_FULL_REFRESH is not True:
    raise ValueError("Review the six target tables, then set ALLOW_FULL_REFRESH = True")
required = ["SOURCE_COLUMNS", "F", "clean_retail", "gold_models"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Run %run ./00_transformations first; missing: {missing}")

raw = (spark.read.option("header", True).option("inferSchema", False)
       .option("multiLine", True).option("escape", '"')
       .option("mode", "FAILFAST").csv(INPUT_PATH))
if raw.columns != SOURCE_COLUMNS:
    raise ValueError("Expected the exact columns exported by scripts/prepare_data.py")
input_rows = raw.count()
if input_rows == 0:
    raise ValueError("Input is empty; no project tables have been overwritten")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {namespace}")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")
bronze = (raw.select("*", F.col("_metadata.file_path").alias("source_file"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_batch", F.lit(INPUT_PATH)))
bronze.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(table("retail_bronze"))
bronze = spark.table(table("retail_bronze"))
if bronze.count() != input_rows:
    raise ValueError("Bronze row count differs from the input preflight; investigate before proceeding")


## Silver and Gold tables

Silver keeps every source row with typed fields and quality flags. Missing customer IDs do not exclude otherwise eligible sales. Duplicate-looking rows remain because the source has no established unique line key.


In [ ]:
silver = clean_retail(bronze)
silver.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable(table('retail_silver'))
silver = spark.table(table('retail_silver'))
silver.filter('NOT eligible').write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable(table('retail_rejects'))
models = gold_models(silver)
for name, frame in models.items():
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable(table(name))

## Reconciliation

Check that input lines equal eligible plus rejected lines, and that daily and product net sales match eligible Silver.


In [ ]:
# Reconciliation is required before interpreting the Gold results.
source_rows = bronze.count()
accepted = silver.filter('eligible').count()
rejected = spark.table(table('retail_rejects')).count()
assert source_rows == silver.count() == accepted + rejected
silver_total = silver.filter('eligible').agg(F.sum('line_value')).first()[0] or Decimal(0)
for name in ['retail_gold_daily', 'retail_gold_product']:
    total = spark.table(table(name)).agg(F.sum('net_sales')).first()[0] or Decimal(0)
    assert total == silver_total, f'{name} failed reconciliation'
print({'input_rows': source_rows, 'accepted_rows': accepted, 'rejected_rows': rejected,
       'eligible_net_sales_GBP': str(silver_total), 'checks': 'row accounting and daily/product totals passed'})
display(silver.groupBy('quality_reasons').count().orderBy(F.desc('count')))
display(spark.table(table('retail_gold_daily')).orderBy('transaction_date'))

## Runtime details

Save the printed versions, input checksum, and Git revision with the run record. Databricks environment version 6, recorded in the earlier export, is separate from the Spark version.


In [ ]:
import platform
print({"python_version": platform.python_version(), "spark_version": spark.version,
       "input_rows": input_rows, "source_path": INPUT_PATH})


## Sales and order cross-check (GBP)

Calculate totals directly from Silver. Negative adjustments reduce net sales but do not establish refunded cash. Costs are absent, so profit is not calculated.


In [ ]:
%sql
SELECT
    currency,
    COUNT(*) AS accepted_rows,
    SUM(CASE WHEN quantity > 0 THEN line_value ELSE 0 END) AS gross_sales,
    SUM(CASE WHEN quantity < 0 THEN -line_value ELSE 0 END) AS return_value,
    SUM(line_value) AS net_sales,
    COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoice_id END) AS sale_orders
FROM retail_silver
WHERE eligible
GROUP BY currency;

## Rejection reasons

Ineligible rows remain available in Silver and the rejects table for review.


In [ ]:
%sql
SELECT
    quality_reasons,
    COUNT(*) AS rejected_rows
FROM retail_rejects
GROUP BY quality_reasons
ORDER BY rejected_rows DESC;

## Repeat purchasing

Count identified customers with at least two sale invoices during the observed input period. This is not a long-term retention measure.


In [ ]:
%sql
SELECT
    currency,
    COUNT(*) AS identified_purchasing_customers,
    SUM(CASE WHEN sale_orders >= 2 THEN 1 ELSE 0 END)
        AS repeat_customers,
    ROUND(
        100.0 * SUM(CASE WHEN sale_orders >= 2 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        2
    ) AS repeat_customer_pct
FROM retail_gold_customer
GROUP BY currency;

## Customer-ID coverage

Measure missing IDs as a share of eligible lines, not customers or sales value.


In [ ]:
%sql
SELECT
    COUNT(*) AS accepted_lines,
    COUNT_IF(customer_id IS NULL) AS lines_without_customer_id,
    ROUND(
        100.0 * COUNT_IF(customer_id IS NULL)
        / NULLIF(COUNT(*), 0), 2
    ) AS missing_customer_id_pct
FROM retail_silver
WHERE eligible;

## Monthly net sales (GBP)

December 2011 ends on December 9. Count monthly sale orders directly from Silver; distinct counts are not generally additive across dates.


In [ ]:
%sql
SELECT
    date_format(transaction_date, 'yyyy-MM') AS month,
    currency,
    SUM(CASE WHEN quantity > 0 THEN line_value ELSE 0 END) AS gross_sales,
    SUM(CASE WHEN quantity < 0 THEN -line_value ELSE 0 END) AS return_value,
    SUM(line_value) AS net_sales,
    COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoice_id END) AS sale_orders
FROM retail_silver
WHERE eligible
GROUP BY 1, 2
ORDER BY 1;


## Leading sales codes

Include source descriptions to distinguish merchandise from postage. Some product codes have several descriptions.


In [ ]:
%sql
WITH top_codes AS (
    SELECT product_id, gross_sales, return_value, net_sales
    FROM retail_gold_product
    ORDER BY net_sales DESC
    LIMIT 10
),
descriptions AS (
    SELECT
        product_id,
        sort_array(collect_set(trim(Description))) AS descriptions
    FROM retail_silver
    WHERE eligible
    GROUP BY product_id
)
SELECT t.*, d.descriptions
FROM top_codes t
LEFT JOIN descriptions d USING (product_id)
ORDER BY t.net_sales DESC;

## Top ten merchandise codes by net sales (GBP)

Exclude the identified postage codes `DOT` and `POST`. All ten entries in the earlier full-data ranking were checked against their descriptions; this is not a classification of every source code.

Code 22423 (REGENCY CAKESTAND 3 TIER) led the full-data run at GBP 164,762.19. Sample results may differ. For the chart, use a horizontal bar with product_id as Y and net_sales as X, leave Group by empty, and disable category sorting to keep the query order.


In [ ]:
%sql
SELECT
    product_id,
    net_sales
FROM retail_gold_product
WHERE product_id NOT IN ('DOT', 'POST')
ORDER BY net_sales DESC
LIMIT 10;

## Description variants

These two codes had multiple descriptions in the top-ten ranking. Grouping by code keeps the variants together.


In [ ]:
%sql
SELECT
    product_id,
    sort_array(collect_set(trim(Description))) AS descriptions
FROM retail_silver
WHERE eligible AND product_id IN ('22502', '22197')
GROUP BY product_id;

## Save the run record

Record the summary, input SHA-256, Git revision, Python/Spark versions, and any failures. Repeat the run against the same input and compare counts and totals.

Publish aggregate results only. Saved charts are in `reports/figures/`; chart settings need to be recreated after import.
